## Imports

In [1]:
%%writefile hull_metric.py
import numpy as np
import pandas as pd
import pandas.api.types

MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates a custom evaluation metric (volatility-adjusted Sharpe ratio).
    """
    if not pandas.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction']

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    # Strategy Sharpe
    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    # Market stats
    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    # Penalties
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


Writing hull_metric.py


In [2]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np
import pandas as pd

# Reemplazar Auto-sklearn por H2O
import h2o
from h2o.automl import H2OAutoML

from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

## Project Directory Structure

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

## Configurations

In [4]:
# ============ PATHS ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                         # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                         # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                # Multiplier of the OLS market forward excess returns predictions to signal 

# ============ MODEL CONFIGS ============
CV: int = 10                                    # Number of cross validation folds in the model fitting
L1_RATIO: float = 0.5                           # ElasticNet mixing parameter
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)    # Constant that multiplies the penalty terms
MAX_ITER: int = 1000000                         # The maximum number of iterations

## Dataclasses Helpers

In [5]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass 
class ElasticNetParameters:
    l1_ratio : float 
    cv: int
    alphas: np.ndarray 
    max_iter: int 
    
    def __post_init__(self): 
        if self.l1_ratio < 0 or self.l1_ratio > 1: 
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

## Set the Parameters

In [6]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

enet_params = ElasticNetParameters(
    l1_ratio = L1_RATIO, 
    cv = CV, 
    alphas = ALPHAS, 
    max_iter = MAX_ITER
)

## Dataset Loading/Creating Helper Functions

In [7]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )

## Converting Return Prediction to Signal

Here is an example of a potential function used to convert a prediction based on the market forward excess return to a daily signal position. 

In [8]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

## Looking at the Data

In [9]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬──────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target   │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---      │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64      │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪══════════╡
│ 8977    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.708599 ┆ 0.004187        ┆ 0.000162       ┆ 0.003713 │
│ 8978    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.725858 ┆ 0.002279        ┆ 0.000162       ┆ 0.001805 │
│ 8979    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.720092 ┆ 0.003541        ┆ 0.000161       ┆ 0.003068 │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴──────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scor

## Generating the Train and Test

In [10]:
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES) 

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler 

## Fitting the Model 

In [11]:
import h2o
from h2o.automl import H2OAutoML

# Inicializar H2O
h2o.init()


# Convertir Polars a pandas antes de concatenar
X_train_pd = X_train.to_pandas()
y_train_pd = y_train.to_pandas()

# Crear H2OFrame para entrenamiento
train_h2o = h2o.H2OFrame(pd.concat([X_train_pd, y_train_pd.rename("target")], axis=1))

# Definir variables
x = X_train.columns
y = "target"

# Entrenar modelo con H2O AutoML
model = H2OAutoML(max_runtime_secs=1800, seed=42)
model.train(x=x, y=y, training_frame=train_h2o)


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "11.0.27" 2025-04-15; OpenJDK Runtime Environment (build 11.0.27+6-post-Ubuntu-0ubuntu122.04); OpenJDK 64-Bit Server VM (build 11.0.27+6-post-Ubuntu-0ubuntu122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.11/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpgq6tci7p
  JVM stdout: /tmp/tmpgq6tci7p/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpgq6tci7p/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,6 months and 16 days
H2O_cluster_name:,H2O_from_python_unknownUser_sa5o2j
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,7.500 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),3/3
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None
Custom metalearner hyperparameters,None


In [12]:
# ==== IMPORTS ====
import numpy as np
import pandas as pd
import h2o
from hull_metric import score, ParticipantVisibleError

# ==== 1) PREPARAR HOLDOUT TEMPORAL ====
# Recomendado: usa una columna temporal si la tienes (p.ej. 'date_id').
# Si no, hacemos un split por orden (último 20% = valid).

# Asumo que tienes el DataFrame base de entrenamiento (con columnas de target y rf) en pandas:
# - Si sólo tienes X_train (features) y y_train (target), vamos a reconstruir un df unido:
train_full_pd = pd.concat([X_train.to_pandas().reset_index(drop=True),
                           y_train.to_pandas().rename("forward_returns").reset_index(drop=True)],
                          axis=1)

# Renombra/asegura la columna de tasa libre de riesgo (ajusta el nombre según tus features)
# por ejemplo si tienes 'rf' en X_train:
if 'risk_free_rate' not in train_full_pd.columns and 'rf' in train_full_pd.columns:
    train_full_pd = train_full_pd.rename(columns={'rf': 'risk_free_rate'})

# Si no tienes rf en features, pon un proxy (no ideal, pero evita romper el flujo)
if 'risk_free_rate' not in train_full_pd.columns:
    train_full_pd['risk_free_rate'] = 0.0

# Split temporal (si existe date_id, ordenamos por él)
if 'date_id' in train_full_pd.columns:
    train_full_pd = train_full_pd.sort_values('date_id').reset_index(drop=True)

# Índice de corte (20% final como valid)
cut = int(len(train_full_pd) * 0.8)
train_pd = train_full_pd.iloc[:cut].reset_index(drop=True)
valid_pd = train_full_pd.iloc[cut:].reset_index(drop=True)

# H2OFrames
train_h2o = h2o.H2OFrame(train_pd.drop(columns=['risk_free_rate'], errors='ignore'))  # H2O entrenará con features + forward_returns
valid_h2o = h2o.H2OFrame(valid_pd.drop(columns=['risk_free_rate'], errors='ignore'))

# Columnas X/Y para H2O (usamos las mismas que en tu entrenamiento original)
y_col = "forward_returns"
x_cols = [c for c in train_h2o.columns if c != y_col]

# ==== 2) OBTENER LEADERBOARD ====
lb = model.leaderboard.as_data_frame()   # DataFrame con 'model_id'
model_ids = lb['model_id'].tolist()

# ==== 3) FUNCIÓN: calibrar k y calcular score ====
def evaluate_model_on_valid(h2o_model, valid_pd, valid_h2o, k_grid=None):
    """
    - Predice en valid
    - Mapea pred->posición: pos = clip(pred*k, 0, 2)
    - Busca k en k_grid que maximiza la métrica
    """
    if k_grid is None:
        # Ajusta el rango si tus predicciones son muy pequeñas/grandes
        k_grid = np.linspace(50, 600, 12)  # 12 valores entre 50 y 600

    # Predicción (regresión) -> columna 'predict'
    pred_valid = h2o_model.predict(valid_h2o).as_data_frame()['predict'].values

    best = {"k": None, "score": -np.inf}
    for k in k_grid:
        positions = np.clip(pred_valid * k, 0, 2)
        # Construir dataframes para la métrica
        solution = valid_pd[['forward_returns', 'risk_free_rate']].copy()
        submission = pd.DataFrame({'prediction': positions})
        try:
            s = score(solution=solution, submission=submission, row_id_column_name='row_id')  # row_id no se usa aquí
            if s > best["score"]:
                best = {"k": k, "score": s}
        except ParticipantVisibleError as e:
            # Si algo falla (p.ej. std=0), lo saltamos
            continue
    return best, pred_valid

# ==== 4) EVALUAR TODOS LOS MODELOS Y RANKEAR ====
results = []
for mid in model_ids:
    m = h2o.get_model(mid)
    best, _ = evaluate_model_on_valid(m, valid_pd, valid_h2o)
    results.append({
        "model_id": mid,
        "best_k": best["k"],
        "metric": best["score"]
    })

rank_df = pd.DataFrame(results).sort_values("metric", ascending=False).reset_index(drop=True)
print(rank_df.head(10))

# ==== 5) ELEGIR EL MEJOR (por tu métrica) ====
best_model_id = rank_df.loc[0, "model_id"]
best_k = rank_df.loc[0, "best_k"]
best_model = h2o.get_model(best_model_id)
print("Mejor por métrica:", best_model_id, " | k =", best_k)


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

...

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████████████| (done) 100%
xgboost prediction progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████████████| (done) 100%
                                           model_id  best_k    metric
0  XGBoost_grid_1_AutoML_1_20251013_230320_model_23   600.0  9.254165
1  XGBoost_grid_1_AutoML_1_20251013_230320_model_19   600.0  9.144336
2   XGBoost_grid_1_AutoML_1_20251013_230320_model_1   600.0  9.031430
3  XGBoost_grid_1_AutoML_1_20251013_230320_model_10   600.0  8.892198
4   XGBoost_grid_1_AutoML_1_20251013_230320_model_3   600.0  8.865949
5   XGBoost_grid_1_AutoML_1_20251013_230320_model_7   600.0  8.759312
6   XGBoost_grid_1_AutoML_1_20251013_230320_model_2   600.0  8.279792
7                XGBoost_1_AutoML_1_20251013_230320   600.0  7.798169
8  XGBoost_grid_1_AutoML_1_20251013_230320_model_14   600.0  7.797829
9   XGBoost_grid_1_AutoML_1_20251013_230320_model_9   600.0  7.510956
Mejor por métrica: XGBoost_grid_1_AutoML_1_20251013_230320_model_23  | k = 600.0


/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [13]:
# después de rankear:
best_model_id = rank_df.loc[0, "model_id"]
best_k = float(rank_df.loc[0, "best_k"])
best_model = h2o.get_model(best_model_id)

# guarda el modelo y el k en /kaggle/working
model_path = h2o.save_model(best_model, path="/kaggle/working", force=True)

import json
meta = {
    "model_path": model_path,
    "best_k": best_k,
    # guarda tus FEATURES exactas y en el mismo orden que entrenaste
    "features": x if isinstance(x, (list, tuple)) else list(x)
}
with open("/kaggle/working/best_meta.json", "w") as f:
    json.dump(meta, f)
print("Guardado:", model_path, "| best_meta.json")


Guardado: /kaggle/working/XGBoost_grid_1_AutoML_1_20251013_230320_model_23 | best_meta.json


In [14]:
import os, json, numpy as np, pandas as pd
import polars as pl
import h2o

# Carga meta (modelo, k, features) una sola vez
with open("/kaggle/working/best_meta.json", "r") as f:
    META = json.load(f)

if not h2o.connection():
    h2o.init()

BEST_MODEL = h2o.load_model(META["model_path"])
FEATURES = META["features"]
BEST_K = float(META["best_k"])

def predict(test: pl.DataFrame) -> float:
    """
    test: polars DataFrame con UNA fila (la del día actual) + potenciales columnas auxiliares.
    Debe transformarse a las FEATURES exactas usadas en entrenamiento.
    """
    # Si ya tienes una función de transformación, úsala aquí:
    # df = create_example_dataset(test)
    # X_row = df.select(FEATURES).to_pandas()

    # Si no, asume que 'test' ya tiene tus FEATURES:
    X_row = test.select(FEATURES).to_pandas()

    # H2OFrame de 1 fila
    X_h2o = h2o.H2OFrame(X_row)

    # Predicción de retorno (o señal cruda)
    y_hat = BEST_MODEL.predict(X_h2o).as_data_frame().values.flatten()[0]

    # Mapear a posición [0, 2] con tu k calibrado
    position = float(np.clip(y_hat * BEST_K, 0.0, 2.0))
    return position


## Prediction Function via Kaggle Server

In [27]:
# def predict(test: pl.DataFrame) -> float:
#     # Renombrar la columna objetivo
#     test = test.rename({'lagged_forward_returns': 'target'})

#     # Crear el dataset de ejemplo con ingeniería de características
#     df: pl.DataFrame = create_example_dataset(test)
#     X_test: pl.DataFrame = df.select(FEATURES)

#     # Convertir de Polars a pandas
#     X_test_pd = X_test.to_pandas()

#     # Convertir a H2OFrame
#     X_test_h2o = h2o.H2OFrame(X_test_pd)

#     # Predecir con el modelo H2OAutoML
#     raw_pred: float = model.predict(X_test_h2o).as_data_frame().values.flatten()[0]

#     # Convertir la predicción en señal de inversión
#     return convert_ret_to_signal(raw_pred, ret_signal_params)

## Launch Server

In [15]:
import kaggle_evaluation

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

import os
DATA_PATH = "/kaggle/input/hull-tactical-market-prediction/"
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    # Esto corre localmente y te genera la submission (según el wrapper de la comp)
    inference_server.run_local_gateway((DATA_PATH,))


/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py:97: RuntimeWarning: 3845 seconds elapsed before server startup.
                This exceeds the startup time limit of 900 seconds that the gateway will enforce
                during the rerun on the hidden test set. Start the server before performing any time consuming steps.
  warnings.warn(


GatewayRuntimeError: (<GatewayRuntimeErrorType.SERVER_RAISED_EXCEPTION: 3>, 'U1\\n\\nResolved plan until failure:\\n\\n\\t---> FAILED HERE RESOLVING \\\'select\\\' <---\\nDF [\\"date_id\\", \\"D1\\", \\"D2\\", \\"D3\\"]; PROJECT */99 COLUMNS", grpc_status:2}')

Closing connection _sid_b56b at exit
H2O session _sid_b56b closed.


In [28]:
# inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     inference_server.serve()
# else:
#     inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%
Parse progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


...

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |

/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


███████████████████████████████████████████| (done) 100%


/usr/local/lib/python3.11/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
